# Financial GraphRAG Engine — Demo Completo

Pipeline end-to-end: Ingesta → Grafo de Conocimiento → Recuperación Híbrida → Generación → Evaluación

**Stack:** Python 3.10+ · SEC EDGAR · bge-m3 · LanceDB · BM25 · Kùzu · Ollama/Groq · LangChain · RAGAS

In [1]:
from __future__ import annotations
import logging, sys, json, os
from pathlib import Path

logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")

os.chdir(Path.cwd().parent)
from src.pipeline import FinancialGraphRAGPipeline
from src.graph.communities import CommunityDetector
from src.llm_factory import create_llm, create_embeddings

In [2]:
# ── Configurar LLM (Ollama local por defecto, Groq como fallback) ──
llm = create_llm()
print(f"LLM ready: {type(llm).__name__} ({llm.model_name if hasattr(llm, 'model_name') else llm.model})")

LLM ready: ChatOllama (qwen2.5:7b)


---
## 1. Ingesta de un Informe 10-K

Descargamos, parseamos y chunkificamos un 10-K real desde SEC EDGAR.

In [3]:
pipeline = FinancialGraphRAGPipeline(llm=llm)

chunks = pipeline.ingest_and_index(
    ticker="AAPL",
    year=2023,
)

print(f"Total chunks generados: {len(chunks)}")
if chunks:
    print(f"  Primer chunk [{chunks[0].section_id}]: {chunks[0].text[:120]}...")

WARNING | Extraction attempt 1 failed for chunk cd165e2f-8a2e-496a-8f66-9e151d5a2558: Expecting property name enclosed in double quotes: line 1 column 110 (char 109)
WARNING | Extraction attempt 2 failed for chunk cd165e2f-8a2e-496a-8f66-9e151d5a2558: Expecting property name enclosed in double quotes: line 1 column 110 (char 109)
WARNING | Extraction attempt 3 failed for chunk cd165e2f-8a2e-496a-8f66-9e151d5a2558: Expecting property name enclosed in double quotes: line 1 column 110 (char 109)
ERROR | All extraction attempts failed for chunk cd165e2f-8a2e-496a-8f66-9e151d5a2558: Expecting property name enclosed in double quotes: line 1 column 110 (char 109)
WARNING | Extraction attempt 1 failed for chunk f5faabb0-e3b8-4a86-81eb-234102a663c6: Expecting ',' delimiter: line 1 column 2643 (char 2642)
WARNING | Extraction attempt 2 failed for chunk f5faabb0-e3b8-4a86-81eb-234102a663c6: Expecting ',' delimiter: line 1 column 2643 (char 2642)
WARNING | Extraction attempt 3 failed for chunk f5f

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Total chunks generados: 68
  Primer chunk [Item 16]: This Annual Report on Form 10-K (“Form 10-K”) contains forward-looking statements, within the meaning of the Private Sec...


### Estructura de un Chunk

In [4]:
if chunks:
    c = chunks[0]
    print(f"chunk_id:       {c.chunk_id}")
    print(f"company_ticker: {c.company_ticker}")
    print(f"fiscal_year:    {c.fiscal_year}")
    print(f"section_id:     {c.section_id}")
    print(f"page_number:    {c.page_number}")
    print(f"token_count:    {c.token_count}")
    print(f"metadata:       {c.metadata}")

chunk_id:       dcb24a1b-86f6-4362-8880-069f0059aa10
company_ticker: AAPL
fiscal_year:    2023
section_id:     Item 16
page_number:    1
token_count:    291
metadata:       {'chunk_seq': '0', 'source': 'AAPL_2023'}


---
## 2. Grafo de Conocimiento

Los chunks ya se insertaron en Kùzu. Ahora inspeccionamos el grafo.

In [5]:
conn = pipeline.graph.schema.connection

result = conn.execute("""
    MATCH (c:DocumentChunk) WHERE c.company_ticker = 'AAPL'
    RETURN c.chunk_id, c.section_id, c.fiscal_year
    LIMIT 5
""")
print("Chunks en Kùzu:")
while result.has_next():
    row = result.get_next()
    print(f"  {row[0][:16]}... | {row[1]} | FY{row[2]}")

Chunks en Kùzu:
  dd6d3eaf-8e08-49... | Item 1 | FY2023
  b0a9b3a4-bcfe-4c... | Item 1 | FY2023
  c0f5bd69-71bb-4c... | Item 1 | FY2023
  a43bb213-ce45-42... | Item 1 | FY2023
  76fe26f5-882c-4a... | Item 1 | FY2023


In [6]:
# ── Extraer tripletes con LLM ──
if chunks:
    triplets = pipeline.graph.extractor.extract_from_chunk(
        chunk_text=chunks[0].text[:2000],
        ticker=chunks[0].company_ticker,
        year=chunks[0].fiscal_year,
        section_id=chunks[0].section_id,
        chunk_id=chunks[0].chunk_id,
    )
    print(f"Tripletes extraídos: {len(triplets)}")
    for t in triplets[:3]:
        print(f"  ({t.source_name}) -[:{t.relation}]-> ({t.target_name})")

Tripletes extraídos: 2
  (AAPL) -[:reported_metric]-> (forward-looking statements)
  (AAPL) -[:impacts_revenue]-> (macroeconomic conditions)


---
## 3. Recuperación Híbrida

### 3a. Búsqueda Vectorial (Dense)

In [7]:
dense_results = pipeline.retrieval.dense.search(
    "What was Apple's revenue in 2023?", top_k=5
)
print(f"Resultados densos: {len(dense_results)}")
for r in dense_results[:3]:
    print(f"  score={r.score:.4f} | {r.text[:100]}...")

Resultados densos: 5
  score=0.5853 | $ | 62,683 | $ | 53,382 Europe: Net sales | $ | 94,294 | $ | 95,118 | $ | 89,307 Operating income | ...
  score=0.5905 | 2022 and 2021 (net income in millions and shares in thousands): 2023 | 2022 | 2021 Numerator: Net in...
  score=0.5935 | Americas | $ | 162,560 | (4) | % | $ | 169,658 | 11 | % | $ | 153,306 Europe | 94,294 | (1) | % | 95...


### 3b. Búsqueda Léxica (BM25)

In [8]:
sparse_results = pipeline.retrieval.sparse.search(
    "risk factors competition supply chain", top_k=5
)
print(f"Resultados BM25: {len(sparse_results)}")
for r in sparse_results[:3]:
    print(f"  score={r.score:.4f} | {r.text[:100]}...")

Resultados BM25: 5
  score=8.5392 | the Company’s ability to maintain a competitive advantage could be materially adversely affected. Th...
  score=7.2152 | The Company’s business, reputation, results of operations, financial condition and stock price can b...
  score=6.5099 | recovery time, experience significant expenditures to resume operations, and lose significant sales....


### 3c. Recorrido en Grafo (Multi-Hop)

In [9]:
graph_results = pipeline.retrieval.graph.search(
    "AAPL revenue segment performance", top_k=5
)
print(f"Resultados de grafo: {len(graph_results)}")
for r in graph_results[:3]:
    print(f"  path={r.traversal_path} | {r.text[:100]}...")

Resultados de grafo: 4
  path=entity_graph_57c378a3 | or goods initiated within an application. From time to time, the Company has made changes to its App...
  path=entity_graph_1e7839d6 | or goods initiated within an application. From time to time, the Company has made changes to its App...
  path=entity_graph_4dc255d3 | or goods initiated within an application. From time to time, the Company has made changes to its App...


### 3d. RRF + Reranking

In [10]:
fused = pipeline.retrieval.fusion.fuse(
    dense=dense_results,
    sparse=sparse_results,
    graph=graph_results,
    top_k=10,
)
print(f"Fusionados (RRF k=60): {len(fused)}")
for f in fused[:5]:
    print(f"  rrf={f.rrf_score:.4f} | d={f.dense_score} s={f.sparse_score} g={f.graph_score}")

reranked = pipeline.retrieval.reranker.rerank(
    query="What was Apple's revenue in 2023?",
    candidates=fused,
    top_k=5,
)
print(f"\nRerankeados: {len(reranked)}")
for r in reranked:
    print(f"  score={r.score:.4f} | {r.text[:80]}...")

Fusionados (RRF k=60): 10
  rrf=0.0164 | d=0.5853265523910522 s=None g=None
  rrf=0.0164 | d=None s=8.539180407159428 g=None
  rrf=0.0164 | d=None s=None g=1.0
  rrf=0.0161 | d=0.5904762744903564 s=None g=None
  rrf=0.0161 | d=None s=7.2151958469518895 g=None


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Rerankeados: 5
  score=0.9846 | $ | 62,683 | $ | 53,382 Europe: Net sales | $ | 94,294 | $ | 95,118 | $ | 89,307...
  score=0.9831 | Americas | $ | 162,560 | (4) | % | $ | 169,658 | 11 | % | $ | 153,306 Europe | 9...
  score=0.9722 | 394,328 | 8 | % | $ | 365,817 (1) Products net sales include amortization of the...
  score=0.9491 | 2022 and 2021 (net income in millions and shares in thousands): 2023 | 2022 | 20...
  score=0.5678 | The Company’s business, reputation, results of operations, financial condition a...


---
## 4. Generación con Citas

Pipeline completo: pregunta → recuperación → RRF → rerank → generar respuesta con citas.

In [11]:
result = pipeline.query("What was Apple's total net revenue for fiscal year 2023?")

print("=" * 70)
print("PREGUNTA:")
print(result.question)
print("\nRESPUESTA:")
print(result.answer)
print("\nCITAS:")
for c in result.citations[:3]:
    print(f"  [{c['company_ticker']} | FY{c['fiscal_year']} | {c['section_id']} | chunk: {c['chunk_id'][:12]}...]")
print(f"\nEstadísticas:")
print(f"  Dense:  {result.dense_results}")
print(f"  Sparse: {result.sparse_results}")
print(f"  Graph:  {result.graph_results}")
print(f"  Final:  {len(result.final_context)} chunks")

PREGUNTA:
What was Apple's total net revenue for fiscal year 2023?

RESPUESTA:
Apple's total net revenue for fiscal year 2023 was $383,285 million.

This can be found in the "Note 2 – Revenue" section of the provided document, specifically in the table that details net sales by category for 2023, where it states:

Total net sales | $ | 383,285 | (3) | % | $ | 394,328 | 8 | % | $ | 365,817

The figure of $383,285 million represents the total net sales for fiscal year 2023.

CITAS:
  [AAPL | FY2023 | Item 7 | chunk: 16be6347-95c...]
  [AAPL | FY2023 | Item 8 | chunk: b7675243-510...]
  [AAPL | FY2023 | Item 8 | chunk: 28d31368-515...]

Estadísticas:
  Dense:  20
  Sparse: 20
  Graph:  0
  Final:  5 chunks


---
## 5. Comunidades de Grafo (Global Search)

Detección de comunidades via Leiden y resúmenes ejecutivos.

In [12]:
detector = CommunityDetector(
    schema=pipeline.graph.schema,
    llm=llm,
)

communities = detector.detect_communities()
print(f"Comunidades detectadas: {len(communities)}")
for cid, members in sorted(communities.items(), key=lambda x: -len(x[1]))[:5]:
    print(f"  Comunidad {cid}: {len(members)} miembros — {members[:3]}...")

Comunidades detectadas: 125
  Comunidad 0: 423 miembros — ['AAPL', 'companies with significant technical, marketing, distribution and other resources', 'companies that have established hardware, software and digital content supplier relationships']...
  Comunidad 1: 51 miembros — ['Apple Inc.', 'other participants in the markets for smartphones, personal computers, tablets, wearables and accessories', 'not specified in the text']...
  Comunidad 2: 22 miembros — ['revenue', 'write-downs on the value of its inventory and other assets', 'purchase commitment cancellation risk']...
  Comunidad 3: 14 miembros — ['results of operations', 'financial condition', 'stock price']...
  Comunidad 4: 10 miembros — ['Total net sales', 'Net sales by reportable segment: Europe', 'Products gross margin']...


In [13]:
summaries = detector.generate_summaries(communities, max_communities=3)
for s in summaries:
    print(f"\n{'='*60}")
    print(f"Comunidad {s.community_id} ({s.entity_count} entidades)")
    print(f"Top: {s.top_entities}")
    print(f"Resumen: {s.summary[:300]}...")


Comunidad 0 (423 entidades)
Top: ['AAPL', 'companies with significant technical, marketing, distribution and other resources', 'companies that have established hardware, software and digital content supplier relationships', 'companies with broader product lines, lower-priced products and a larger installed base of active devices', 'companies that provide content to users for free']
Resumen: Based on the provided information, I will identify and categorize relationships between entities where applicable. Here are some key relationships:

1. **Financial Metrics and Financial Statements:**
   - "Note 7 to the financial statements" is related to "Non-current Marketable Debt Securities - Du...

Comunidad 1 (51 entidades)
Top: []
Resumen: ### Strategic Summary for Community ID: 1

#### Key Entities:
- **None Identified**: The provided community does not include any specific entities or companies that form the core of this business community.

#### Material Risks:
- **Lack of Data**: Without

---
## 6. Evaluación con RAGAS

Corremos las 4 métricas sobre el dataset de prueba.

In [ ]:
import subprocess, sys

# RAGAS usa asyncio y choca con el event loop de Jupyter (especialmente en Python 3.14).
# Cerramos el pipeline (libera el lock de Kùzu) y lanzamos la evaluación en un proceso
# independiente, donde ragas corre sin conflictos.
pipeline.close()

result = subprocess.run(
    [sys.executable, "evals/run_ragas_eval.py", "--samples", "3", "--output", "evals/results/demo_report.json"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


Muestras de evaluación: 15


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

ERROR | Exception in callback <_asyncio.TaskStepMethWrapper object at 0x770d4e4a3ca0>()
handle: <Handle <_asyncio.TaskStepMethWrapper object at 0x770d4e4a3ca0>()>
Traceback (most recent call last):
  File "/usr/lib/python3.14/asyncio/events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: Cannot enter into task <Task pending name='Task-106' coro=<_async_in_context.<locals>.run_in_context() running at /home/daniel/Projects/financial-graphrag/venv/lib/python3.14/site-packages/ipykernel/utils.py:57> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/daniel/Projects/financial-graphrag/venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py:563]> while another task <Task pending name='Task-105' coro=<Kernel.shell_main() running at /home/daniel/Projects/financial-graphrag/venv/lib/python3.14/site-packages/ipykernel/kernelbase.py:621> cb=[Task.task_wakeup()]> is being executed.
ERROR | Task w

Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]

ERROR | Exception in callback Task.__step()
handle: <Handle Task.__step()>
Traceback (most recent call last):
  File "/usr/lib/python3.14/asyncio/events.py", line 94, in _run
    self._context.run(self._callback, *self._args)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: cannot enter context: <_contextvars.Context object at 0x7f1118248f80> is already entered
ERROR | Task was destroyed but it is pending!
task: <Task pending name='Task-27' coro=<_async_in_context.<locals>.run_in_context() running at /home/daniel/Projects/financial-graphrag/venv/lib/python3.14/site-packages/ipykernel/utils.py:60> wait_for=<Task pending name='Task-40' coro=<Kernel.shell_main() running at /home/daniel/Projects/financial-graphrag/venv/lib/python3.14/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callback.<locals>._log_error() at /home/daniel/Projects/financial-graphrag/venv/lib/python3.14/site-packages/zmq/eventloop/zmqstream.py:563]>
/usr/lib/python3.1

In [ ]:
print("Scores individuales:")
for r in results:
    print(f"\n  Q: {r.sample.question[:60]}...")
    for metric, score in r.scores.items():
        print(f"    {metric:25s} {score:.4f}")

---
## Limpieza

In [ ]:
pipeline.close()
print("Pipeline cerrado.")